Env setup and import

In [4]:
import torch
from torch import nn
from torchrl.collectors import MultiSyncCollector
from torchrl.data.replay_buffers import ReplayBuffer
from torchrl.data.replay_buffers.samplers import SamplerWithoutReplacement
from torchrl.data.replay_buffers.storages import LazyTensorStorage
from torchrl.objectives import ClipPPOLoss, ValueEstimators
from env_simplified import make_env
from ai_setup import make_policy_critic

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)
env = make_env()
policy, critic = make_policy_critic(env, 'policy_checkpoint.pth', 'critic_checkpoint.pth')

cuda


c:\Users\ctc73\PycharmProjects\MahjongAI\.venv\Lib\site-packages\torchrl\envs\libs\pettingzoo.py:281: UserWarning: PettingZoo in TorchRL is tested using version == 1.24.3 , If you are using a different version and are experiencing compatibility issues,please raise an issue in the TorchRL github.
  warnings.warn(


Loss function and optimizer

In [5]:
policy = policy.to(device)
loss_module = ClipPPOLoss(
    actor_network=policy, # type: ignore
    critic_network=critic,
    entropy_coeff=0.01
)
loss_module.set_keys(  # We have to tell the loss where to find the keys
    reward=env.reward_key,
    action=env.action_key,
    value=("agents", "state_value"),
    # These last 2 keys will be expanded to match the reward shape
    done=("agents", "done"),                # per-agent
    terminated=("agents", "terminated"),
)
gamma = 0.995  # discount factor
lmbda = 0.9  # lambda for generalised advantage estimation
lr = 2e-4
loss_module.make_value_estimator(
    ValueEstimators.GAE, gamma=gamma, lmbda=lmbda
)  
GAE = loss_module.value_estimator

optim = torch.optim.Adam(loss_module.parameters(), lr)

loss_module = loss_module.to(device)

Train loop

In [ ]:
num_epochs = 5
max_grad_norm = 0.1
frames_per_batch = 2048  # Number of team frames collected per training iteration
n_iters = 180 # Number of sampling and training iterations
total_frames = frames_per_batch * n_iters
minibatch_size = 256

if __name__ == "__main__":
    replay_buffer = ReplayBuffer(
        storage=LazyTensorStorage(
            frames_per_batch, device=device
        ),  # We store the frames_per_batch collected at each iteration
        sampler=SamplerWithoutReplacement(),
        batch_size=minibatch_size,  # We will sample minibatches of this siz
    )
    policy=policy.to(device)
    collector = MultiSyncCollector(
        [make_env] * 12,
        policy=policy,
        device='cpu',
        storing_device=device,
        frames_per_batch=frames_per_batch,
        total_frames=total_frames,
        cat_results=0,
    )
    from tqdm.auto import tqdm
    for it, tensordict_data in enumerate(tqdm(collector)):
        tensordict_data.set(
            ("next", "agents", "done"),
            tensordict_data.get(("next", "done"))
            .unsqueeze(-1)
            .expand(tensordict_data.get_item_shape(("next", env.reward_key))),
        )
        tensordict_data.set(
            ("next", "agents", "terminated"),
            tensordict_data.get(("next", "terminated"))
            .unsqueeze(-1)
            .expand(tensordict_data.get_item_shape(("next", env.reward_key))),
        )
        # We need to expand the done and terminated to match the reward shape (this is expected by the value estimator)

        with torch.no_grad():
            GAE(
                tensordict_data,
                params=loss_module.critic_network_params,
                target_params=loss_module.target_critic_network_params
            )  # Compute GAE and add it to the data

        data_view = tensordict_data.reshape(-1)  # Flatten the batch size to shuffle data
        replay_buffer.extend(data_view)

        for _ in range(num_epochs):
            for _ in range(frames_per_batch // minibatch_size):
                subdata = replay_buffer.sample()
                loss_vals = loss_module(subdata)

                loss_value = (
                    loss_vals["loss_objective"]
                    + loss_vals["loss_critic"]
                    + loss_vals["loss_entropy"]
                )

                loss_value.backward()

                torch.nn.utils.clip_grad_norm_(
                    loss_module.parameters(), max_grad_norm
                )  # Optional

                optim.step()
                optim.zero_grad()

        collector.update_policy_weights_()


In [ ]:
torch.save(policy.state_dict(), 'policy_7.pth')
torch.save(critic.state_dict(), 'critic_7.pth')

Rollout per step

In [6]:
import pygame
from sys import exit
from pygame_visualizer import render_game_state
from torchrl.envs.utils import step_mdp  # <-- Added missing utility

pygame.init()
screen = pygame.display.set_mode(size=(800, 800))
font = pygame.font.Font("C:/Windows/Fonts/seguisym.ttf", 48)
torch.set_printoptions(precision=4, sci_mode=False)
td = env.reset()
policy = policy.to('cpu').eval()

while True:
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            pygame.quit()
            exit()

        # Step through the game manually by pressing SPACE
        if event.type == pygame.KEYDOWN and event.key == pygame.K_SPACE:
            # 1. Check if the game is already over before stepping
            if td.get("done", torch.tensor([False])).any():
                print("Episode finished! Resetting environment...")
                td = env.reset()
                continue
                
            # 2. Policy reads root "observation" and writes root "action" into td
            td = policy(td)
            print(torch.softmax(td[('agents', 'logits')], dim=1))
            
            # 3. Environment executes the action and creates the "next" sub-TensorDict
            td = env.step(td)
            
            # 4. Check if this new step ended the game
            if td[("next", "done")].any():
                print(f"Game Over! Reward: {td[('next', 'agents', 'reward')]}")
            
            # 5. Move "next" keys to root level so the policy can read them next turn
            td = step_mdp(td)
            
            
    screen.fill('white')
    # Render the internal unwrapped environment game state
    render_game_state(env._env.gamestate, screen, font)
    pygame.display.update()

tensor([[0.0114, 0.0135, 0.0099, 0.0109, 0.0134, 0.0122, 0.0122, 0.0142, 0.0179,
         0.0174, 0.0148, 0.0153, 0.0112, 0.0129, 0.0152, 0.0119, 0.0130, 0.0105,
         0.0117, 0.0130, 0.0146, 0.0148, 0.0136, 0.0111, 0.0141, 0.0113, 0.0131,
         0.0113, 0.0138, 0.0150, 0.0135, 0.0154, 0.0115, 0.0139, 0.0130, 0.0168,
         0.0134, 0.0141, 0.0159, 0.0170, 0.0141, 0.0109, 0.0107, 0.0110, 0.0139,
         0.0127, 0.0121, 0.0167, 0.0134, 0.0116, 0.0128, 0.0118, 0.0123, 0.0130,
         0.0131, 0.0158, 0.0124, 0.0195, 0.0113, 0.0138, 0.0133, 0.0137, 0.0134,
         0.0126, 0.0124, 0.0145, 0.0123, 0.0115, 0.0130, 0.0148, 0.0107, 0.0169,
         0.0142, 0.0136, 0.0108],
        [0.0103, 0.0133, 0.0108, 0.0165, 0.0148, 0.0106, 0.0121, 0.0114, 0.0118,
         0.0111, 0.0139, 0.0138, 0.0105, 0.0123, 0.0154, 0.0128, 0.0116, 0.0106,
         0.0126, 0.0139, 0.0146, 0.0169, 0.0135, 0.0116, 0.0121, 0.0159, 0.0162,
         0.0125, 0.0122, 0.0146, 0.0151, 0.0123, 0.0116, 0.0138, 0.0109, 0.

SystemExit: 